Import Statements

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

Input data and LSTM parameters/hyperparameters

In [ ]:
#Get Dataset
dataset = pd.read_csv('../../data/sandbox/stocks/demo_stock_aapl.csv')
#Data feature headers
data = dataset[['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']].values

#Normalize data
normalize = MinMaxScaler()
normalized_data = normalize.fit_transform(data)

#LSTM model hyperparameters
input_size = 6 #Number of used input features
hidden_size = 64
num_layers = 2
sequence_length = 30
learning_rate = 0.001
epochs = 50
output_size = 1 #Number of output features
num_forecasted_days = 7 #Number of days to forecast prices

Create LSTM model

In [ ]:
#Create the model
class LSTM(nn.Module):
    #Initialize the LSTM model
    def __init__(self, input_size=6, hidden_size=64, num_layers=2, output_size=1):
        super(LSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.hidden_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        output, states = self.lstm(x)
        return self.hidden_layer(output[:, -1, :])

Train LSTM model on data

In [ ]:
lstm = LSTM(input_size, hidden_size, num_layers, output_size=1)
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm.parameters(), lr=learning_rate)

def create_sequences(data, sequence_length):
    windows = len(data) - sequence_length
    xs = [data[i : i + sequence_length] for i in range(windows)]
    ys = [data[i + sequence_length, 3] for i in range(windows)]
    return torch.tensor(xs, dtype=torch.float32), torch.tensor(ys, dtype=torch.float32).unsqueeze(1)


X_train, y_train = create_sequences(normalized_data, sequence_length)

lstm.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = lstm(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

Output data/results/conclusions

In [ ]:
def prediction(model, input_sequence, normalize):
    model.eval()
    with torch.no_grad():
        scaled_prediction = model(input_sequence).cpu().numpy()
        flattened_prediction = np.zeros((len(scaled_prediction), input_size))
        flattened_prediction[:, 3] = scaled_prediction.flatten()
        inverse_array = normalize.inverse_transform(flattened_prediction)
        return inverse_array[:, 3]


recent_data = normalized_data[-30:] 
latest_data_sequence = torch.tensor(recent_data, dtype=torch.float32).unsqueeze(0)
predicted_price = prediction(lstm, latest_data_sequence, normalize)


def forecast_future(model, last_sequence, scaler, days_to_predict=7):
    model.eval()
    current_sequence = last_sequence.clone()
    predictions_scaled = []
    with torch.no_grad():
        for i in range(days_to_predict):
            prediction = model(current_sequence)
            predictions_scaled.append(prediction.item())
            avg_volume = current_sequence[0, :, 5].mean().item()
            new_row = torch.tensor([[prediction.item()] * 5 + [avg_volume]], dtype=torch.float32).unsqueeze(0)
            current_sequence = torch.cat((current_sequence[:, 1:, :], new_row), dim=1)
    scaled_predictions = np.zeros((days_to_predict, 6))
    scaled_predictions[:, 3] = predictions_scaled
    return scaler.inverse_transform(scaled_predictions)[:, 3]

future_prices = forecast_future(lstm, latest_data_sequence, normalize, num_forecasted_days)
